# Sonifying the Heart: inner emotion detection from ECG through music

*Music Workshop project · Chervith Reddy (2024101076) · Manidhar Sukasi (2023101067) · Sathwik Reddy (2024121002)*

A heartbeat already has a rhythm, and that rhythm is steered by the same nervous system that moves with our emotions. This notebook is the complete pipeline that turns a real electrocardiogram (ECG) into a short piece of music. It then reads the emotion expressed by that music, as a window into the person’s inner state:

```
ECG data ──► Heartbeats ──► Music ──► Emotion
```

- **A steady heart** becomes a calm piece: one melody note held over a gentle chord progression, with an even “lub-dub” pulse.
- **An irregular heart**, here atrial fibrillation (AFib), becomes a restless one: the melody escapes the scale, the chords sour and the pulse stumbles.

Our own music-emotion model reads each piece, and two independent pretrained models give second and third opinions.

| § | Step | Stage | Supporting code |
|---|---|---|---|
| 1 | The ECG data | 1. ECG data | `ecgmusic/data.py` |
| 2 | From ECG to heartbeats | 2. Heartbeats | `ecgmusic/data.py` |
| 3 | The melody: one note per heartbeat | 3. Music | `ecgmusic/melody.py` |
| 4 | The emotion estimate, read from the melody | 4. Emotion | `ecgmusic/emotion.py` |
| 5 | The arrangement: harmony and pulse | 3. Music | `ecgmusic/arrangement.py` |
| 6 | MIDI, audio and the score | 3. Music | `ecgmusic/audio.py`, `ecgmusic/plots.py` |
| 7 | The full experiment: 14 pieces from 7 recordings | all | `ecgmusic/pipeline.py` |
| 8 | Results | all | |
| 9 | Second and third opinions: pretrained emotion models | 4. Emotion | `ecgmusic/essentia_models.py`, `ecgmusic/music2emo_crosscheck.py` |
| 10 | Using your own ECG file | all | `ecgmusic/pipeline.py` |
| 11 | Limitations and ethics | | |
| 12 | References | | |

The emotion (§4) comes before the arrangement (§5) because the emotion read from the melody chooses the arrangement’s melody instrument.

Every parameter lives in `ecgmusic/config.py`. The same pipeline runs from the command line (`python -m ecgmusic ...`, see [`README.md`](README.md)). The design, all results and the validation plan are written up in [`project_description.md`](project_description.md), and [`Progress.md`](Progress.md) is the dated log of the work.

## 0. Setup

Run the notebook from top to bottom. The four switches below control the expensive steps. **All of them are off by default**, so the notebook runs on a small machine:

| Switch | When `True` | Cost |
|---|---|---|
| `DOWNLOAD_DATA` | downloads both PhysioNet databases into `data/` | about 720 MB |
| `RENDER_AUDIO` | renders WAV audio with FluidSynth and plays it here | needs FluidSynth; seconds per piece |
| `RUN_ESSENTIA` | runs the pretrained Essentia models on the audio (needs `RENDER_AUDIO`) | TensorFlow; slow on small machines |
| `REBUILD_EXAMPLES` | recomputes all 14 example pieces and the result tables, even if `output/` already has them | 1–2 minutes, more with the options above |

**With everything off**, the notebook streams the records it needs from PhysioNet, composes them, and draws their scores.

- **First run:** it builds the 14 example pieces (MIDI and scores only) into `output/`.
- **Later runs:** they reuse those pieces.
- **Pretrained models:** their results appear once their tables exist; §9 explains how to make them.

Setup: `pip install -r requirements.txt`, plus `sudo apt install fluidsynth fluid-soundfont-gm` for audio.

In [ ]:
DOWNLOAD_DATA = False
RENDER_AUDIO = False
RUN_ESSENTIA = False
REBUILD_EXAMPLES = False

%matplotlib inline
import shutil
import tempfile
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pretty_midi
import wfdb
from IPython.display import Audio, display

from ecgmusic import config
from ecgmusic.data import download_database, record_clips
from ecgmusic.emotion import descriptors, estimate_emotion
from ecgmusic.melody import deviation_to_pitch, melody
from ecgmusic.pipeline import analyze_signal, build_examples, compose, save
from ecgmusic.plots import plot_circumplex, plot_clip, plot_score

pd.set_option('display.precision', 2)
DEMO_DIR = Path(tempfile.mkdtemp(prefix='ecgmusic_'))  # scratch output, outside the repository
HAS_FLUIDSYNTH = shutil.which('fluidsynth') is not None
print('FluidSynth installed:', HAS_FLUIDSYNTH)
if RENDER_AUDIO and not HAS_FLUIDSYNTH:
    raise RuntimeError('RENDER_AUDIO needs FluidSynth: sudo apt install fluidsynth fluid-soundfont-gm')
if RUN_ESSENTIA and not RENDER_AUDIO:
    print('RUN_ESSENTIA listens to rendered audio, so it is skipped while RENDER_AUDIO is False.')

## 1. The ECG data

Both datasets come from [PhysioNet](https://physionet.org/), the open archive of physiological recordings maintained by MIT. They are the two databases named in our proposal:

| | MIT-BIH Atrial Fibrillation Database (`afdb`) | MIT-BIH Arrhythmia Database (`mitdb`) |
|---|---|---|
| Recordings | 25 long-term recordings of about 10 hours (23 contain the ECG signal) | 48 half-hour recordings from 47 people |
| People | adults with atrial fibrillation, mostly paroxysmal; no per-person demographics published | 25 men aged 32–89 and 22 women aged 23–89; about 60% hospital inpatients |
| Recorded | Beth Israel Hospital, Boston, 1983 | Beth Israel Hospital, Boston, 1975–1979 |
| Sampling | 250 Hz, 2 ECG leads | 360 Hz, 2 ECG leads |
| Heartbeat annotations | `.qrs`, found by an automatic detector and **never checked by hand** | inside `.atr`, checked by two or more cardiologists |
| Rhythm annotations | `.atr`, labelled by experts: `N` normal, `AFIB`, `AFL` flutter, `J` junctional | `+` marks inside `.atr`: `N`, `AFIB` and many other arrhythmias |

`afdb` is the main source. Everyone in it has AFib, so most records contain both a normal stretch and an AFib stretch. `mitdb` focuses on other arrhythmias, and only 3 of its 48 records contain at least 30 seconds of both rhythms.

**Nothing has to be downloaded.** If a record is not in `data/`, the code streams only what it needs from PhysioNet: the annotations plus 30 seconds of signal. That is a few hundred kilobytes per record, instead of 720 MB for both databases.

Who these recordings come from matters before using this tool for anyone else; see [`project_description.md` §14](project_description.md#14-where-the-data-comes-from-and-ethics).

In [ ]:
if DOWNLOAD_DATA:
    for db in ('afdb', 'mitdb'):
        download_database(db)

for db in ('afdb', 'mitdb'):
    on_disk = len(list((config.DATA_DIR / db).glob('*.hea')))
    print(f'{db}: {on_disk} records on disk' + ('' if on_disk else ', so records are streamed from PhysioNet'))

## 2. From ECG to heartbeats

Every heartbeat shows up in the ECG as a sharp spike, the **R-peak**. The time between two consecutive R-peaks is the **R-R interval**, about 0.6–1.0 s for a resting heart. It is the most important number in this project: the rhythm of the music *is* the sequence of R-R intervals.

How steady that rhythm is gets measured with **RMSSD**, the standard beat-to-beat variability measure:

$$\mathrm{RMSSD} = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N-1}\left(RR_{i+1} - RR_i\right)^2}$$

A resting heart in normal rhythm varies by a few tens of milliseconds at most. During AFib the upper chambers quiver chaotically, the lower chambers beat at irregular moments, and RMSSD jumps to well over 100 ms.

### Choosing the 30-second clips

For each record we cut one 30-second **normal** clip and one 30-second **AFib** clip out of the stretches the experts labelled. Inside every labelled stretch, up to 5 evenly spaced candidate windows are scored:

- **Normal (`N`)**: keep the *steadiest* window, the one with the lowest RMSSD.
- **AFib (`AFIB`)**: keep the window with the *most detected heartbeats*.

The reason is a data-quality problem we found the hard way. `afdb`’s heartbeat marks come from an automatic detector that was never corrected by hand, and in places it jitters. Our first version simply took the first 30 seconds of each stretch, and for two records (`04015`, `06995`) the “normal” clip came out *more* irregular than the AFib clip: RMSSD 199 ms against 105 ms for `04015`. The waveform showed a perfectly steady rhythm with misplaced beat marks. Real sinus rhythm is low-variability, so preferring the steadiest window steps around those glitches. In AFib the irregularity is genuine, so we only ask for the best-detected window and never pick the most dramatic one.

In [ ]:
clips = record_clips('04043', 'afdb')
for label, clip in clips.items():
    print(f'{label:>4}: {len(clip.peaks)} heartbeats, mean R-R {clip.mean_rr_ms:.0f} ms, RMSSD {clip.rmssd_ms:.1f} ms')

In [ ]:
for label, clip in clips.items():
    plot_clip(clip, f'afdb 04043, {label} clip: RMSSD {clip.rmssd_ms:.1f} ms')
plt.show()

The difference is already visible: the normal clip’s heartbeats (red crosses) are evenly spaced, while the gaps in the AFib clip keep changing. Its RMSSD is about 16 times higher (134.9 ms against 8.5 ms).

## 3. The melody: one note per heartbeat

Each heartbeat becomes one note, following the rules of our proposal:

| From the ECG | Becomes |
|---|---|
| the moment of the heartbeat | the moment the note starts; the tempo is never quantised, so every note lands exactly when the heart beat |
| the R-R interval | the note’s length: 85% of the interval, leaving a small breath before the next note |
| how this interval differs from the recent average | the note’s pitch |
| the ECG amplitude at the heartbeat | the note’s loudness: MIDI velocity 40–110, scaled within each clip |

**Pitch.** For beat $i$ the *deviation* compares its interval with the average of the last five intervals, itself included (fewer at the very start):

$$d_i = \frac{RR_i - \overline{RR}_i}{\overline{RR}_i}, \qquad \overline{RR}_i = \operatorname{mean}\left(RR_{i-4}, \dots, RR_i\right)$$

- $d_i = 0$, a beat exactly on time, plays the root note **C4**.
- A **faster** beat ($d_i < 0$) moves the melody **up**; a **slower** beat ($d_i > 0$) moves it **down**.
- While $|d_i| < 12\%$ the melody stays on the **C-major pentatonic scale** (C D E G A), stepping up or down along it. Pentatonic melodies sound consonant over almost any chord, which is a large part of why a steady heart sounds calm.
- From $|d_i| \ge 12\%$ the melody leaves the scale and moves **chromatically**, a semitone at a time. A deviation of 50% or more reaches the extreme, two octaves from C4.

The number of steps away from C4 is $\operatorname{round}\left(\min(|d_i| / 0.5,\ 1) \times n \times 2\right)$, where $n$ is the size of the scale in use: 5 for pentatonic, 12 for chromatic.

> **Two bugs fixed during development.** An earlier version sent faster beats *down*, the opposite of this design. Its downward “pentatonic” steps also mirrored the intervals (B♭, A♭) instead of walking down the scale (A, G), so a steady heart actually played a whole-tone fragment that clashes with C-major chords. Both are fixed; the table below checks the behaviour.

In [ ]:
rows = []
for d in (-0.30, -0.09, -0.07, -0.02, 0.0, 0.02, 0.07, 0.09, 0.30):
    pitch = deviation_to_pitch(d)
    beat = 'faster' if d < 0 else 'slower' if d > 0 else 'on time'
    rows.append({'R-R deviation': f'{d:+.0%}', 'beat': beat,
                 'note': pretty_midi.note_number_to_name(pitch), 'MIDI pitch': pitch})
pd.DataFrame(rows)

In [ ]:
melodies = {label: melody(clip) for label, clip in clips.items()}
for label, notes in melodies.items():
    pitches = [n.pitch for n in notes]
    low, high = (pretty_midi.note_number_to_name(p) for p in (min(pitches), max(pitches)))
    share = 100 * np.mean([n.chromatic for n in notes])
    print(f'{label:>4}: {len(notes)} notes from {low} to {high}, {share:.0f}% chromatic')

pd.DataFrame([asdict(n) for n in melodies['AFIB'][:8]])

## 4. The emotion estimate

Music psychology often places emotions on **Russell’s circumplex** (1980). It has two axes, **valence** (unpleasant ↔ pleasant) and **arousal** (calm ↔ energetic). Each quadrant holds a family of emotions: happy/excited, tense/anxious, sad/depressed and calm/relaxed.

Which musical features carry which emotions is summarised by **Gabrielsson & Lindström (2010)**: tempo and loudness raise arousal, consonance raises valence, and an irregular rhythm adds tension. We place each melody on the plane from three such cues:

| Cue | Measured as | Normalised over |
|---|---|---|
| rhythmic irregularity $\hat I$ | spread of the gaps between notes (standard deviation / mean) | 0 to 0.30 |
| pitch spread $\hat S$ | standard deviation of the pitches, in semitones | 0 to 11 |
| consonance $\hat C$ | share of the notes that stayed on the pentatonic scale | 0.30 to 1.0 |

$$\text{arousal} = 2\left(0.55\,\hat I + 0.45\,\hat S - 0.5\right) \qquad \text{valence} = 2\left(0.70\,\hat C + 0.30\,(1 - \hat S) - 0.5\right)$$

Both land between −1 and 1. The normalisation ranges were calibrated on the 14 example clips.

**Tempo and loudness are deliberately left out**, although they are the classic arousal cues:

- **Tempo** varies more between people than between rhythms. Record 201’s AFib clip, at 101 beats per minute, is *slower* than record 04043’s normal clip at 108. So a clip’s tempo alone cannot place it.
- **Loudness** is re-scaled within every clip, so it carries no information across clips at all.

The quadrant also chooses the **melody instrument** of the arrangement (§5):

| Quadrant | Instrument (General MIDI) |
|---|---|
| calm / relaxed: + valence, − arousal | Acoustic Guitar (nylon) |
| tense / anxious: − valence, + arousal | Distortion Guitar |
| happy / excited: + valence, + arousal | Marimba |
| sad / depressed: − valence, − arousal | Cello |

> **From the music’s emotion to the person’s.** This estimate describes the emotional character of the *generated music*. The project’s aim is to read it as a window into the person’s inner state, because the nervous system that steers the heartbeat also moves with emotion.
>
> That step has not been validated yet. These recordings come with no emotion reports, and atrial fibrillation is an electrical problem in the heart, not an emotion. A calm person can have AFib, and an anxious person can have a perfectly regular heartbeat. How to test the step is set out in [`project_description.md` §13](project_description.md#13-validation-plan).

In [ ]:
for label, notes in melodies.items():
    emotion = estimate_emotion(notes)
    cues = {name: round(value, 3) for name, value in descriptors(notes).items()}
    print(f'{label:>4}: {emotion.quadrant:<14} valence {emotion.valence:+.2f}, arousal {emotion.arousal:+.2f}'
          f' -> melody on {emotion.instrument}')
    print(f'      cues: {cues}')

## 5. The arrangement: harmony and pulse

A single melodic line is a sonification; a Music Workshop project should be a piece of music. The arrangement adds two parts under the melody, and both are still driven entirely by the ECG.

**Harmony, on strings.** Every 4 heartbeats get one sustained chord, following the progression **C – Am – F – G** (I – vi – IV – V in C major), voiced below the melody:

| Chord | Notes |
|---|---|
| C | C3 E3 G3 |
| Am | A2 C3 E3 |
| F | F2 A2 C3 |
| G | G2 B2 D3 |

The colour of each chord depends on how irregular its 4 beats were, measured by their average absolute deviation:

| Average deviation | Chord played | Sound |
|---|---|---|
| below 12% | the progression’s own chord | consonant |
| 12% to 25% | a diminished triad on the same root | tense, the classic suspense chord of film music |
| 25% and above | a cluster of root, minor second and tritone | chaotic |

The thresholds were calibrated on the real clips. In normal rhythm the 4-beat average never went above 5.5%, so the harmony always stays consonant. In AFib the median is 14%: 61% of the chords turn tense or worse, and the most irregular 11% become clusters. When an AFib stretch briefly steadies, the progression comes back, as if the music keeps trying to return to calm.

A steady heart has an elegant consequence: its melody sits on C4 almost all the time, so it becomes a **pedal tone**, one held note while the harmony moves underneath, a classic compositional device. Because the pentatonic scale fits every chord of the progression, every in-scale note sounds consonant.

**Pulse, on drums.** Every heartbeat also triggers a soft **“lub-dub”** on two bass drums, the sound a stethoscope hears. The “lub” falls on the R-peak; the “dub” follows after 35% of that beat’s interval (0.3 s at most), at 60% of the loudness. In normal rhythm the pulse is perfectly even; in AFib it stumbles.

**Mix.** Measured by rendering each part alone for record 04043: the strings sit about 7–9 dB below the melody and the pulse about 12–17 dB below it, so the melody leads and the pulse is felt more than heard.

In [ ]:
def note_names(pitches):
    return ' '.join(pretty_midi.note_number_to_name(p) for p in pitches)

pieces = {label: compose(clip) for label, clip in clips.items()}
for label, piece in pieces.items():
    tiers = [c.tier for c in piece.chords]
    counts = ', '.join(f'{tiers.count(t)} {t}' for t in ('consonant', 'tense', 'chaotic'))
    print(f'{label:>4}: melody on {piece.emotion.instrument}; {len(tiers)} chords: {counts}')

pd.DataFrame([{'from (s)': c.start, 'to (s)': c.end, 'chord': note_names(c.pitches), 'tier': c.tier}
              for c in pieces['AFIB'].chords[:8]])

## 6. MIDI, audio and the score

Each piece is written twice:

- **`melody.mid`**: the melody alone on a piano, the raw sonification with nothing added.
- **`arrangement.mid`**: the full piece, with the melody on the instrument chosen by the emotion estimate, the strings and the pulse.

The MIDI files are rendered to audio by [FluidSynth](https://www.fluidsynth.org/) with the FluidR3 General MIDI soundfont, then normalised so that the loudest moment reaches 85% of full scale; a raw render of such sparse parts peaks at only about 5%.

The **score** puts everything on one time axis: the ECG with its heartbeats, each beat’s deviation (the dashed lines at ±12% mark where the melody leaves the scale), and the generated notes and chords.

In [ ]:
for label, piece in pieces.items():
    plot_score(clips[label], piece.notes, piece.chords,
               f'afdb 04043 {label}: {piece.emotion.quadrant}, melody on {piece.emotion.instrument}')
plt.show()

In the normal clip the melody is a straight line on C4, the pedal tone, over the green, consonant progression. In the AFib clip the melody scatters (red notes are chromatic) and most chords turn orange (diminished). The two drops to C2 line up with the two longest pauses, deviations of +54% and +59%: when the heart skips, the melody falls.

In [ ]:
for label, piece in pieces.items():
    files = save(piece, clips[label], DEMO_DIR / f'04043_{label}', f'afdb 04043 {label}', audio=RENDER_AUDIO)
    print(f'{label:>4}:', ', '.join(path.name for path in files.values() if path), 'in', files['score_png'].parent)
    if RENDER_AUDIO:
        display(Audio(str(files['arrangement_wav'])))

if not RENDER_AUDIO:
    for label in ('N', 'AFIB'):
        saved = config.EXAMPLES_DIR / f'04043_{label}' / 'arrangement.wav'
        if saved.exists():
            print(f'Saved example, {label}:')
            display(Audio(str(saved)))

## 7. The full experiment: 14 pieces from 7 recordings

The same steps run for 7 recordings: five `afdb` records, plus the two usable `mitdb` records with at least 30 seconds of both rhythms. Each gives a normal and an AFib piece:

| Database | Records |
|---|---|
| `afdb` | 04015, 04043, 04126, 06995, 08455 |
| `mitdb` | 201, 202 |

The 7 recordings come from 6 people, because `mitdb` 201 and 202 were taken from the same tape of one 68-year-old man.

**Why `mitdb` record 219 is left out.** It also contains both rhythms, but even its steadiest “normal” window has an RMSSD of 270.6 ms. That is higher than its own AFib clip (145.9 ms). The annotations explain why: 133 *non-conducted P-waves* in 30 minutes, atrial beats blocked before they reach the ventricles. That is a genuine conduction irregularity that the rhythm label “normal” does not capture.

Every method in this notebook flags the record on its own:

- Its normal clip is estimated *tense* and its AFib clip *calm*.
- Its normal clip gets tense and chaotic chords (3 consonant / 2 tense / 2 chaotic), while its AFib clip is mostly consonant (9 / 1 / 0).
- Essentia hears its AFib piece as *less* energetic than its normal one, the opposite of every other record.

Including it would misrepresent what a normal heart sounds like.

`build_examples()` writes one folder per piece to `output/examples/<record>_<rhythm>/`: `melody.mid`, `arrangement.mid`, `score.png`, and the WAV files when audio is rendered. It also writes the result tables in `output/`. The next cell builds them the first time, or whenever `REBUILD_EXAMPLES` is on. Otherwise it only loads the saved summary table.

In [ ]:
SUMMARY_CSV = config.OUTPUT_DIR / 'examples_summary.csv'
if REBUILD_EXAMPLES or not SUMMARY_CSV.exists():
    build_examples(audio=RENDER_AUDIO, essentia=RUN_ESSENTIA and RENDER_AUDIO)

summary = pd.read_csv(SUMMARY_CSV, dtype={'record': str})
summary

## 8. Results

### Does a healthy heart sound different from AFib?

Side by side, heart by heart:

In [ ]:
normal = summary[summary['label'] == 'N'].set_index('record')
afib = summary[summary['label'] == 'AFIB'].set_index('record')
columns = ['rmssd_ms', 'pct_chromatic', 'pitch_std', 'quadrant']
pd.concat({'normal': normal[columns], 'AFib': afib[columns]}, axis=1)

In [ ]:
tier_columns = ['consonant_chords', 'tense_chords', 'chaotic_chords']
print('RMSSD, AFib / normal:', (afib.rmssd_ms / normal.rmssd_ms).round(1).to_dict())
print('Chords under normal pieces:', normal[tier_columns].sum().to_dict())
print('Chords under AFib pieces:  ', afib[tier_columns].sum().to_dict())
print('Estimated emotion:', summary.groupby('label').quadrant.value_counts().to_dict())

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(normal))
left.bar(x - 0.2, normal.rmssd_ms, 0.4, color='tab:blue', label='normal')
left.bar(x + 0.2, afib.rmssd_ms, 0.4, color='tab:red', label='AFib')
left.set(xticks=x, xticklabels=normal.index, ylabel='RMSSD (ms)', title='Beat-to-beat variability')
left.legend()
bottom = np.zeros(len(afib))
for column, color in zip(tier_columns, ('tab:green', 'tab:orange', 'tab:purple')):
    right.bar(afib.index, afib[column], bottom=bottom, color=color, label=column.split('_')[0])
    bottom += afib[column].to_numpy()
right.set(ylabel='chords', title='Harmony of the AFib pieces (normal pieces: all consonant)')
right.legend()
plt.tight_layout()
plt.show()

Yes, in every one of the 7 recordings:

- **Rhythm.**
  - AFib RMSSD is 3 to 17 times the same recording’s normal RMSSD (normal 7.6–64.1 ms, AFib 99.9–194.3 ms).
  - The two `mitdb` normal clips, both from the same man, vary more (37–64 ms) than the `afdb` ones (8–19 ms). Within each recording the gap is always large.
- **Melody.**
  - Every normal melody stays entirely on the pentatonic scale (0% chromatic notes), with a pitch spread (standard deviation) of at most 2 semitones.
  - Every AFib melody leaves the scale for 31–66% of its notes, with a pitch spread of 6–11 semitones.
- **Harmony.** All 67 chords under the normal pieces are consonant. Under the AFib pieces, 68 of 111 chords turn tense (56) or chaotic (12).
- **Emotion estimate.** All 7 normal pieces come out calm/relaxed, and 6 of 7 AFib pieces tense/anxious.
  - The exception, `08455`, is a genuinely mild AFib stretch. It has the fewest chromatic notes (31%) and the smallest pitch spread of all AFib clips.
  - It sits at the centre of the plane (valence +0.07, arousal +0.02), and its harmony stays mostly consonant (11 / 4 / 0).
  - The arrangement reports that honestly instead of exaggerating it.

In [ ]:
plot_circumplex(summary.to_dict('records'))
plt.show()

Each arrow runs from a recording’s normal piece (blue circle, bottom right: calm) to the same recording’s AFib piece (red triangle). Six of the seven point up and to the left into tense/anxious; the arrow that stops near the centre belongs to `08455`.

## 9. Second and third opinions: pretrained emotion models

Our emotion estimate is a formula we designed, so we also asked two independent, openly published models what *they* hear. Both were trained on real music by other researchers and never saw anything from this project.

- **[Essentia](https://essentia.upf.edu/models.html)**, from the Music Technology Group at Universitat Pompeu Fabra.
  - **Four mood classifiers** (happy, sad, relaxed, aggressive) on Discogs-EffNet audio embeddings. Each classifier outputs the probability of its mood and of the opposite. The order differs between models (`happy, non_happy` but `non_sad, sad`), so `config.MOOD_COLUMNS` records which column holds the mood.
  - **A valence/arousal regressor** trained on the DEAM dataset, on a 1–9 scale where 5 is neutral.
  - It listens to both the plain melody and the full arrangement. Its models (about 25 MB) download automatically on first use.
- **[music2emo](https://github.com/AMAAI-Lab/Music2Emotion)** (Kang & Herremans, 2025): a newer model built on MERT embeddings, giving valence/arousal (1–9) and mood tags. It needs older library versions than this project, so it runs from its own environment (see the README).

Both listen to audio. This section shows their result tables once they exist:

- **Essentia’s** table is written by §7 when `RENDER_AUDIO` and `RUN_ESSENTIA` are on.
- **music2emo’s** table is written by `python -m ecgmusic.music2emo_crosscheck`, run from its own environment.

In [ ]:
if RUN_ESSENTIA and RENDER_AUDIO:
    from ecgmusic.essentia_models import analyze_wav
    for label in clips:
        live = analyze_wav(DEMO_DIR / f'04043_{label}' / 'melody.wav')
        print(f'{label:>4} (live):', {name: round(value, 3) for name, value in live.items()})

ESSENTIA_CSV = config.OUTPUT_DIR / 'essentia_results.csv'
essentia = pd.read_csv(ESSENTIA_CSV) if ESSENTIA_CSV.exists() else None
if essentia is None:
    print('No Essentia results yet: set RENDER_AUDIO, RUN_ESSENTIA and REBUILD_EXAMPLES to True and run again.')
else:
    essentia[['record', 'label']] = essentia['name'].str.rsplit('_', n=1, expand=True)
    display(essentia.pivot_table(index=['record', 'label'], columns='render',
                                 values=[*config.MOODS, 'valence', 'arousal']))

In [ ]:
def afib_minus_normal(frame):
    # per record: the AFib value minus the normal value, for every numeric column
    frame = frame.set_index(['record', 'label']).select_dtypes('number')
    return frame.xs('AFIB', level='label') - frame.xs('N', level='label')

if essentia is not None:
    for render in ('melody', 'arrangement'):
        delta = afib_minus_normal(essentia[essentia['render'] == render].drop(columns=['name', 'render']))
        print(f'{render:>11}: records where AFib scores higher:', (delta > 0).sum().to_dict())

What Essentia hears, in our run of the 14 pieces (all numbers in `project_description.md` §11.2):

- **Every piece sounds mostly calm.** “Relaxed” is the strongest mood of all 28 renders. For every plain melody, the DEAM valence and arousal stay below the neutral 5.
- **On the plain melody, every AFib piece sounds livelier than the same recording’s normal piece**, on all six measures, in all 7 recordings (42 of 42). Compared with the normal piece, the AFib piece is:
  - happier (0.10–0.14 against 0.04–0.06);
  - more aggressive (0.14–0.26 against 0.05–0.12);
  - less sad (0.37–0.53 against 0.62–0.71);
  - less relaxed (0.68–0.82 against 0.88–0.92);
  - higher in valence and arousal.
- **Essentia agrees that AFib is more energetic**, as our estimate does (higher arousal). **But it also hears AFib as more pleasant** (higher valence), the opposite of our estimate.
  - One likely reason is tempo. Every AFib clip is also the faster heart (101–160 beats per minute, against 50–108), and faster music is classically heard as more energetic and often happier. Our estimate leaves tempo out on purpose.
  - Models trained on DEAM are also known to learn valence and arousal as correlated.
- **The arrangement widens the gap.**
  - The normal arrangements become the calmest pieces of all (relaxed 0.97–0.98, sad 0.90–0.93, happy and aggressive at most 0.03).
  - Six of the seven AFib arrangements sound 7–15 times more “aggressive” (0.12–0.20 against 0.01–0.02) and 7–18 times happier than the normal ones.
  - The exception is again the mild `08455`, whose arrangement got a marimba and mostly consonant chords.
- **Caveat:** this agreement is partly circular, because the distortion guitar was *chosen* by our estimate’s “tense” label. It shows that the arrangement communicates the intended character, not that the estimate was right.

In [ ]:
MUSIC2EMO_CSV = config.OUTPUT_DIR / 'music2emo_results.csv'
music2emo = pd.read_csv(MUSIC2EMO_CSV) if MUSIC2EMO_CSV.exists() else None
three_way = None
if essentia is None or music2emo is None:
    print('The three-way comparison needs both pretrained-model tables: Essentia from §7 with its switches on, '
          'music2emo from `python -m ecgmusic.music2emo_crosscheck` in its own environment (README §9.7).')
else:
    music2emo[['record', 'label']] = music2emo['name'].str.rsplit('_', n=1, expand=True)
    axes = ['record', 'label', 'valence', 'arousal']
    three_way = pd.concat({
        'our estimate': afib_minus_normal(summary[axes]),
        'Essentia': afib_minus_normal(essentia[essentia['render'] == 'melody'][axes]),
        'music2emo': afib_minus_normal(music2emo[axes]),
    }, axis=1)
    display(three_way)

In [ ]:
if three_way is not None:
    display((three_way > 0).sum().unstack().rename_axis(index='records where AFib is higher'))

The three-way table shows AFib minus normal, so a positive number means “AFib scores higher”:

| | valence (pleasantness) | arousal (energy) |
|---|---|---|
| our estimate | AFib lower in 7 of 7 recordings | AFib higher in 7 of 7 |
| Essentia | AFib **higher** in 7 of 7 | AFib higher in 7 of 7 |
| music2emo | AFib lower in 6 of 7 | AFib **lower** in 6 of 7 |

All three methods agree that a normal and an AFib heart produce music that sounds different, in every recording. But no two of them agree on *which* emotion describes the difference on both axes:

- Our estimate and music2emo agree on valence.
- Our estimate and Essentia agree on arousal.
- music2emo’s exception is record `201`, where it hears the AFib piece as both more pleasant and more energetic.

This is not a flaw to hide. Weak agreement between emotion-recognition models is a well-known open problem in music-emotion research. Each model is trained on different datasets (DEAM, emoMusic, PMEmo, MTG-Jamendo) with different listeners and instructions, and none of them has ever heard an ECG turned into music.

**The robust conclusion so far:** the music makes the difference between a steady and an irregular heart audible, and every emotion model hears it. Two questions remain open:

- Which emotion does that difference stand for?
- Does it match what the person actually felt?

They are the open question of this project. [`project_description.md`](project_description.md) §12–13 discuss them and set out how to test them with ECG datasets that come with people’s own emotion ratings (WESAD, DREAMER, AMIGOS).

## 10. Using your own ECG file

Once there is a signal, nothing depends on PhysioNet’s annotations: `analyze_signal()` finds the heartbeats itself with the **XQRS** detector from the `wfdb` package, then composes the music exactly as above. Below it gets two 30-second windows of record 04043 *without* their annotation files:

- the first 30 seconds give 53 detected heartbeats, a mean R-R of 559 ms and an RMSSD of 14.2 ms, against 54 annotated beats, 559 ms and 13.8 ms for almost the same stretch in the database’s own annotations, and come out calm;
- a window inside the AFib stretch, from 1070 s, gives an RMSSD of 205.8 ms and comes out tense.

For your own recordings use `analyze_file()`, which reads a WFDB record (path without extension) or a plain file of samples, one per line (then give `fs`, the sampling rate):

```python
from ecgmusic.pipeline import analyze_file
result = analyze_file('path/to/record')            # WFDB record
result = analyze_file('path/to/ecg.csv', fs=250)   # one sample per line
```

From the command line: `python -m ecgmusic analyze path/to/ecg.csv --fs 250`. Results go to `output/analysis/<name>/`, which is kept out of git because a real person’s recording is personal health data.

In [ ]:
local = config.DATA_DIR / 'afdb' / '04043'
source = {'record_name': str(local)} if Path(f'{local}.hea').exists() else {'record_name': '04043', 'pn_dir': 'afdb'}
fs = 250
for start in (0, 1070):
    signal = wfdb.rdrecord(sampfrom=start * fs, sampto=(start + 30) * fs, **source).p_signal[:, 0]
    result = analyze_signal(signal, fs, name=f'04043_from_{start}s', duration=30,
                            out_dir=DEMO_DIR / f'own_file_{start}', audio=RENDER_AUDIO)
    clip, piece = result['clip'], result['piece']
    print(f'from {start:>4} s: {len(clip.peaks)} heartbeats found, mean R-R {clip.mean_rr_ms:.0f} ms, '
          f'RMSSD {clip.rmssd_ms:.1f} ms -> {piece.emotion.quadrant}')

## 11. Limitations and ethics

- **Not a medical tool.** This is a proof of concept for listening and research; nothing here was validated for diagnosis.
- **The emotion is read from the music, and has not yet been validated against the person’s feelings.** AFib is an electrical problem in the heart, so “tense” here means the *generated music* sounds tense, which our mapping was designed to produce. Testing the step from the music’s emotion to the person’s needs ECG recordings with emotion reports ([`project_description.md` §13](project_description.md#13-validation-plan)).
- **A small, specific dataset.** 14 clips from 7 recordings of 6 adult heart patients (`mitdb` 201 and 202 come from the same man), recorded in one Boston hospital between 1975 and 1983. The emotion estimate’s ranges were calibrated on exactly these clips.
- **Automatic heartbeat marks.** `afdb`’s beat annotations were never corrected by hand. The clip selection works around their glitches but cannot remove every one; record 04126’s AFib clip, for example, shows visible baseline wander.
- **Using this with any other group of people**, for example as a non-verbal window into the emotional state of autistic people, needs its own validation, consent and ethical review. Research on heart-rate variability in autism exists, and it works directly from measures such as RMSSD. [`project_description.md` §14](project_description.md#14-where-the-data-comes-from-and-ethics) summarises it and what a responsible study would need.

## 12. References

**Data**
- Goldberger AL, et al. PhysioBank, PhysioToolkit, and PhysioNet. *Circulation* 2000;101(23):e215–e220. [doi:10.1161/01.CIR.101.23.e215](https://doi.org/10.1161/01.CIR.101.23.e215)
- Moody GB, Mark RG. A new method for detecting atrial fibrillation using R-R intervals. *Computers in Cardiology* 1983;10:227–230. [MIT-BIH Atrial Fibrillation Database](https://physionet.org/content/afdb/1.0.0/)
- Moody GB, Mark RG. The impact of the MIT-BIH Arrhythmia Database. *IEEE Engineering in Medicine and Biology* 2001;20(3):45–50. [doi:10.1109/51.932724](https://doi.org/10.1109/51.932724) · [MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0/)

**Music and emotion**
- Russell JA. A circumplex model of affect. *Journal of Personality and Social Psychology* 1980;39(6):1161–1178.
- Gabrielsson A, Lindström E. The role of structure in the musical expression of emotions. In: Juslin PN, Sloboda JA (eds), *Handbook of Music and Emotion*. Oxford University Press, 2010.
- Aljanaki A, Yang Y-H, Soleymani M. Developing a benchmark for emotional analysis of music. *PLOS ONE* 2017 (the DEAM dataset).

**Models and tools**
- Bogdanov D, et al. Essentia: an audio analysis library for music information retrieval. *ISMIR* 2013. [essentia.upf.edu](https://essentia.upf.edu/), [pretrained models](https://essentia.upf.edu/models.html)
- Alonso-Jiménez P, Serra X, Bogdanov D. Music representation learning based on editorial metadata from Discogs. *ISMIR* 2022 (the Discogs-EffNet embeddings).
- Kang J, Herremans D. Towards unified music emotion recognition across dimensional and categorical models. 2025. [arXiv:2502.03979](https://arxiv.org/abs/2502.03979) · [code](https://github.com/AMAAI-Lab/Music2Emotion)
- [wfdb-python](https://github.com/MIT-LCP/wfdb-python) (record reading and the XQRS heartbeat detector), [pretty_midi](https://github.com/craffel/pretty-midi), [FluidSynth](https://www.fluidsynth.org/)